# Evaluate Service-Aware Scheduling

Maps satellites to the service categories used by Bor et al. and tests service-aware scheduling against unweighted adaptive scheduling.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import math
import os
import subprocess
import sys
import tempfile
import time
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import gammaln

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


RAW_DIR = PROJECT_ROOT / 'ordem_raw'
REFINEMENT_ZIP = RAW_DIR / 'job-b9824543-c020-42b7-b747-61b298670ce4.zip'
OLD_ACTION_DIR = PROJECT_ROOT / 'phase1_outputs' / 'ordem_action_library' / '33aa70ca_80_plus_baseline'
OLD_RESPONSE_PATH = OLD_ACTION_DIR / 'ordem_action_response_matrix.csv.gz'
FINE_GRID_DIR = PROJECT_ROOT / 'phase1_outputs' / 'ordem_action_library' / 'b982_fine_elevation_49'
FINE_RESPONSE_PATH = FINE_GRID_DIR / 'ordem_fine_elevation_response_matrix_49.csv'
EVENT_PATH = PROJECT_ROOT / 'phase0_outputs' / 'catalog_matching' / 'ca_cfar_1p75_24h_final_audited' / 'final_consolidated_catalog_association.csv'
DETECTOR_DIR = PROJECT_ROOT / 'phase0_outputs' / 'detector_benchmark_raw_swerling_v2'
SELECTED_DETECTOR_PATH = DETECTOR_DIR / 'SELECTED_DETECTOR.json'
PREVIOUS_CONFIG_CANDIDATES = sorted(
    (PROJECT_ROOT / 'phase3_outputs' / 'final_bayesian_policy').glob('fine_grid_*/final_experiment_config.json')
)
DYNAMIC_DIR = PROJECT_ROOT / 'phase2_outputs' / 'dynamic_asset_exposure' / 'active_omm_action_audit_v1'
SNAPSHOT_DIR = DYNAMIC_DIR / 'snapshots'
CSWIM_DIR = PROJECT_ROOT / 'inputs' / 'cswim_v1'
CSWIM_CLASSIFIED_PATH = CSWIM_DIR / 'classified_satellites.parquet'
CSWIM_COST_PATH = CSWIM_DIR / 'fleet_costs.csv'
CSWIM_SOURCE_PATH = CSWIM_DIR / 'C_SWIM_V1_SOURCE_AND_MAPPING.json'

PREP_DIR = PROJECT_ROOT / 'phase3_outputs' / 'service_weighted_validation' / '_response_preparation'
PREP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = PREP_DIR                                                                                   

EXPECTED_REFINEMENT_SHA256 = 'eade76d0fcca361e662b8a5da48b67baa809530d3e028b825da445752e75a10a'
RANGE_MIN_KM = 1026.639
RANGE_MAX_KM = 1084.050
ALTITUDE_EDGES = np.arange(915.0, 1065.0 + 0.1, 15.0)
ALTITUDE_CENTERS = 0.5 * (ALTITUDE_EDGES[:-1] + ALTITUDE_EDGES[1:])
ELEVATIONS = np.arange(65.0, 89.0 + 0.01, 0.5)
N_SHELLS = len(ALTITUDE_CENTERS)
N_ACTIONS = len(ELEVATIONS)
CENTER_ACTION = int(np.where(np.isclose(ELEVATIONS, 75.0))[0][0])
BUDGET = int(os.environ.get('DEBRIS_SERVICE_BUDGET', '24'))
N_PRIMARY = int(os.environ.get('DEBRIS_SERVICE_PRIMARY_REPS', '300'))
N_SENSITIVITY = int(os.environ.get('DEBRIS_SERVICE_SENS_REPS', '100'))
BOOTSTRAP_DRAWS = int(os.environ.get('DEBRIS_SERVICE_BOOTSTRAP_DRAWS', '5000'))
CHECKPOINT_EVERY = int(os.environ.get('DEBRIS_SERVICE_CHECKPOINT_EVERY', '10'))
FORCE_RECOMPUTE = os.environ.get('DEBRIS_SERVICE_FORCE', '0') == '1'
MASTER_SEED = 20260831
DWELL_HOURS = 1.0
PROPAGATION_STEP_MINUTES = 15
PROPAGATION_HOURS = 24
SERVICE_CATEGORIES = ('NAV', 'COMMS', 'WEATHER', 'EO', 'MIL')

SIZE_CASES = {
    'gt_1mm': {'suffix': '30', 'old_column': 'window_contribution_gt_1mm_m2_yr', 'label': '>1 mm'},
    'gt_3p16mm': {'suffix': '35', 'old_column': 'window_contribution_gt_3p16mm_m2_yr', 'label': '>3.16 mm'},
    'gt_1cm': {'suffix': '40', 'old_column': 'window_contribution_gt_1cm_m2_yr', 'label': '>1 cm'},
    'gt_3p16cm': {'suffix': '45', 'old_column': 'window_contribution_gt_3p16cm_m2_yr', 'label': '>3.16 cm'},
    'gt_10cm': {'suffix': '50', 'old_column': 'window_contribution_gt_10cm_m2_yr', 'label': '>10 cm'},
}

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def stable_seed(*parts):
    payload = '|'.join(map(str, parts)).encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], 'little') % (2**32 - 1)

required = [
    REFINEMENT_ZIP, OLD_RESPONSE_PATH, FINE_RESPONSE_PATH, EVENT_PATH,
    SELECTED_DETECTOR_PATH, CSWIM_CLASSIFIED_PATH, CSWIM_COST_PATH, CSWIM_SOURCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing frozen input files:\n' + '\n'.join(missing))
if not PREVIOUS_CONFIG_CANDIDATES:
    raise FileNotFoundError('No completed fine-grid Bayesian configuration was found.')
preferred_previous_config = (
    PROJECT_ROOT / 'phase3_outputs' / 'final_bayesian_policy'
    / 'fine_grid_4fa819596ffc' / 'final_experiment_config.json'
)
PREVIOUS_CONFIG_PATH = (
    preferred_previous_config if preferred_previous_config.is_file()
    else PREVIOUS_CONFIG_CANDIDATES[-1]
)
snapshots = sorted(SNAPSHOT_DIR.glob('active_omm_*.csv'))
if not snapshots:
    raise FileNotFoundError('The cached CelesTrak OMM snapshot from the active-satellite audit is missing.')
SNAPSHOT_PATH = snapshots[-1]

if sha256_file(REFINEMENT_ZIP) != EXPECTED_REFINEMENT_SHA256:
    raise RuntimeError('The completed b982 ORDEM ZIP checksum changed.')

events = pd.read_csv(EVENT_PATH)
event_counts = events.association_tier.value_counts().to_dict()
expected_counts = {'unresolved': 106, 'tier_a_strict': 46, 'tier_b_empirical': 3}
if len(events) != 155 or event_counts != expected_counts:
    raise RuntimeError(f'Frozen event decomposition changed: {len(events)}, {event_counts}')
detector = json.loads(SELECTED_DETECTOR_PATH.read_text())
previous_config = json.loads(PREVIOUS_CONFIG_PATH.read_text())
FALSE_RATE_PER_HOUR = float(detector['selection']['false_tracks_per_hour_upper95'])
UNRESOLVED_RATE_PER_HOUR = event_counts['unresolved'] / 24.0
NET_UNRESOLVED_SIGNAL_RATE = max(UNRESOLVED_RATE_PER_HOUR - FALSE_RATE_PER_HOUR, 0.0)
INFERENCE_DISPERSION = float(previous_config['negative_binomial_inference_dispersion'])
PRIOR_LOG_SD = float(previous_config['prior_log_sd'])
PRIOR_CORRELATION_LENGTH_KM = float(previous_config['prior_correlation_length_km'])
PRIOR_NUGGET_FRACTION = float(previous_config['prior_nugget_fraction'])

print('research root:', PROJECT_ROOT)
print('Frozen active snapshot:', SNAPSHOT_PATH.name)
print('Primary / sensitivity worlds:', N_PRIMARY, '/', N_SENSITIVITY)
print('CPU runtime is correct; GPU is not needed.')


## 1. Reproduce C-SWIM categories and propagate the frozen active catalog


In [ ]:
try:
    import pyarrow              
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
try:
    from sgp4.api import Satrec, jday
    from sgp4 import omm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sgp4'])
    from sgp4.api import Satrec, jday
    from sgp4 import omm

cswim_source = json.loads(CSWIM_SOURCE_PATH.read_text())
classified = pd.read_parquet(CSWIM_CLASSIFIED_PATH)
fleet_costs = pd.read_csv(CSWIM_COST_PATH)
classified['NORAD_CAT_ID'] = pd.to_numeric(classified.NORAD_CAT_ID, errors='coerce').astype('Int64')
fleet_costs['satcat'] = pd.to_numeric(fleet_costs.satcat, errors='coerce').astype('Int64')

def cswim_service(primary, constellation):
                                                                    
    if constellation in {'GOES', 'JPSS'}:
        return 'WEATHER'
    if constellation == 'Spire':
        return 'EO'
    if primary == 'Navigation & Positioning':
        return 'NAV'
    if primary == 'Communications & Broadcasting':
        return 'COMMS'
    if primary == 'Earth Observation & Remote Sensing':
        return 'EO'
    if primary == 'Military & Intelligence':
        return 'MIL'
    if pd.isna(primary):
        return 'UNCLASSIFIED'
    return 'OTHER'

fleet_costs['service'] = [
    cswim_service(primary, constellation)
    for primary, constellation in zip(fleet_costs.classification_primary, fleet_costs.constellation)
]
reproduced_counts = fleet_costs.service.value_counts().to_dict()
expected_service_counts = cswim_source['frozen_c_swim_count_check']
if reproduced_counts != expected_service_counts:
    raise RuntimeError(
        'C-SWIM service mapping does not reproduce the frozen source counts. '
        f'Observed {reproduced_counts}; expected {expected_service_counts}'
    )

lookup = classified[['NORAD_CAT_ID', 'classification_primary']].merge(
    fleet_costs[['satcat', 'constellation', 'cost_estimate_M']],
    left_on='NORAD_CAT_ID', right_on='satcat', how='left', validate='one_to_one',
)
lookup['service'] = [
    cswim_service(primary, constellation)
    for primary, constellation in zip(lookup.classification_primary, lookup.constellation)
]
lookup = lookup.drop(columns=['satcat']).drop_duplicates('NORAD_CAT_ID')

active_omm = pd.read_csv(SNAPSHOT_PATH, dtype={'NORAD_CAT_ID': str, 'OBJECT_ID': str})
active_omm['NORAD_CAT_ID_int'] = pd.to_numeric(active_omm.NORAD_CAT_ID, errors='coerce').astype('Int64')
active = active_omm.merge(
    lookup, left_on='NORAD_CAT_ID_int', right_on='NORAD_CAT_ID', how='left',
    suffixes=('', '_cswim'), validate='many_to_one',
)
active['service'] = active.service.fillna('UNCLASSIFIED')
active.loc[active.classification_primary.isna(), 'service'] = 'UNCLASSIFIED'

snapshot_metadata_paths = sorted(SNAPSHOT_DIR.glob('active_omm_*.metadata.json'))
if snapshot_metadata_paths:
    snapshot_metadata = json.loads(snapshot_metadata_paths[-1].read_text())
    propagation_start = pd.Timestamp(snapshot_metadata['retrieved_utc'])
else:
    propagation_start = pd.Timestamp(SNAPSHOT_PATH.stat().st_mtime, unit='s', tz='UTC')
if propagation_start.tzinfo is None:
    propagation_start = propagation_start.tz_localize('UTC')
else:
    propagation_start = propagation_start.tz_convert('UTC')
propagation_start = propagation_start.floor(f'{PROPAGATION_STEP_MINUTES}min')
samples_per_hour = 60 // PROPAGATION_STEP_MINUTES
sample_count = PROPAGATION_HOURS * samples_per_hour
sample_times = pd.date_range(
    propagation_start, periods=sample_count,
    freq=f'{PROPAGATION_STEP_MINUTES}min', tz='UTC',
)
seconds = sample_times.second.to_numpy(float) + sample_times.microsecond.to_numpy(float) / 1e6
jd, fr = jday(
    sample_times.year.to_numpy(), sample_times.month.to_numpy(), sample_times.day.to_numpy(),
    sample_times.hour.to_numpy(), sample_times.minute.to_numpy(), seconds,
)

MU_EARTH_KM3_S2 = 398600.4418
EARTH_RADIUS_KM = 6378.137

def mean_element_fallback(fields):
    mean_motion_rev_day = float(fields['MEAN_MOTION'])
    eccentricity = float(fields['ECCENTRICITY'])
    inclination = np.deg2rad(float(fields['INCLINATION']))
    argument_pericenter = np.deg2rad(float(fields['ARG_OF_PERICENTER']))
    mean_anomaly_epoch = np.deg2rad(float(fields['MEAN_ANOMALY']))
    epoch = pd.Timestamp(fields['EPOCH'])
    if epoch.tzinfo is None:
        epoch = epoch.tz_localize('UTC')
    else:
        epoch = epoch.tz_convert('UTC')
    delta_days = (sample_times - epoch).total_seconds().to_numpy() / 86400.0
    mean_anomaly = np.mod(mean_anomaly_epoch + 2 * np.pi * mean_motion_rev_day * delta_days, 2 * np.pi)
    eccentric_anomaly = mean_anomaly.copy()
    for _ in range(8):
        eccentric_anomaly -= (
            eccentric_anomaly - eccentricity * np.sin(eccentric_anomaly) - mean_anomaly
        ) / np.maximum(1 - eccentricity * np.cos(eccentric_anomaly), 1e-10)
    mean_motion_rad_s = mean_motion_rev_day * 2 * np.pi / 86400.0
    semimajor_axis = (MU_EARTH_KM3_S2 / mean_motion_rad_s ** 2) ** (1 / 3)
    radius = semimajor_axis * (1 - eccentricity * np.cos(eccentric_anomaly))
    altitude = radius - EARTH_RADIUS_KM
    valid = np.isfinite(altitude) & (altitude > 80) & (altitude < 100000)
    return altitude, valid

raw_rows = list(csv.DictReader(io.StringIO(SNAPSHOT_PATH.read_text())))
raw_by_id = {str(row['NORAD_CAT_ID']).strip(): row for row in raw_rows}
service_index = {service: index for index, service in enumerate(SERVICE_CATEGORIES)}
category_shell_samples = np.zeros((len(SERVICE_CATEGORIES), N_SHELLS), float)
category_shell_value = np.zeros_like(category_shell_samples)
other_shell_samples = np.zeros(N_SHELLS, float)
unclassified_shell_samples = np.zeros(N_SHELLS, float)
all_shell_samples = np.zeros(N_SHELLS, float)
orbit_rows = []
propagation_failures = []

for row in active.itertuples(index=False):
    catalog_id = str(row.NORAD_CAT_ID).strip()
    fields = raw_by_id.get(catalog_id)
    if fields is None:
        propagation_failures.append({'NORAD_CAT_ID': catalog_id, 'reason': 'raw OMM row missing'})
        continue
    altitude = valid = None
    engine = 'sgp4'
    try:
        satellite = Satrec()
        omm.initialize(satellite, fields)
        errors, positions, _ = satellite.sgp4_array(jd, fr)
        radii = np.linalg.norm(positions, axis=1)
        altitude = radii - EARTH_RADIUS_KM
        valid = (errors == 0) & np.isfinite(altitude)
    except Exception:
        altitude = valid = None
    if altitude is None:
        try:
            altitude, valid = mean_element_fallback(fields)
            engine = 'mean_element_fallback'
        except Exception as exc:
            propagation_failures.append({'NORAD_CAT_ID': catalog_id, 'reason': repr(exc)})
            continue
    if valid.sum() == 0:
        propagation_failures.append({'NORAD_CAT_ID': catalog_id, 'reason': 'no valid samples'})
        continue
    shell = np.searchsorted(ALTITUDE_EDGES, altitude, side='right') - 1
    in_window = valid & (shell >= 0) & (shell < N_SHELLS)
    if in_window.any():
        np.add.at(all_shell_samples, shell[in_window], 1.0)
        if row.service in service_index:
            idx = service_index[row.service]
            np.add.at(category_shell_samples[idx], shell[in_window], 1.0)
            cost = float(row.cost_estimate_M) if pd.notna(row.cost_estimate_M) and row.cost_estimate_M > 0 else 1.0
            np.add.at(category_shell_value[idx], shell[in_window], cost)
        elif row.service == 'OTHER':
            np.add.at(other_shell_samples, shell[in_window], 1.0)
        else:
            np.add.at(unclassified_shell_samples, shell[in_window], 1.0)
    orbit_rows.append({
        'OBJECT_NAME': row.OBJECT_NAME,
        'NORAD_CAT_ID': catalog_id,
        'classification_primary': row.classification_primary,
        'constellation': row.constellation,
        'service': row.service,
        'cost_estimate_M': row.cost_estimate_M,
        'propagation_engine': engine,
        'valid_samples': int(valid.sum()),
        'samples_in_window': int(in_window.sum()),
        'fraction_in_window': float(in_window.sum() / valid.sum()),
        'minimum_altitude_km': float(np.nanmin(altitude[valid])),
        'maximum_altitude_km': float(np.nanmax(altitude[valid])),
    })

orbit_audit = pd.DataFrame(orbit_rows)
failure_audit = pd.DataFrame(propagation_failures)
if len(orbit_audit) < 0.9 * len(active_omm):
    raise RuntimeError('Fewer than 90% of active satellites propagated successfully.')

in_window = orbit_audit.loc[orbit_audit.samples_in_window > 0].copy()
classified_in_window = in_window.service.isin(SERVICE_CATEGORIES + ('OTHER',))
main_service_in_window = in_window.service.isin(SERVICE_CATEGORIES)
satellite_match_rate = float(classified_in_window.mean()) if len(in_window) else 0.0
sample_match_rate = float(
    (category_shell_samples.sum() + other_shell_samples.sum()) / max(all_shell_samples.sum(), 1.0)
)

print('C-SWIM MAPPING AND ORBIT AUDIT PASSED')
print('  Frozen C-SWIM counts reproduced:', reproduced_counts)
print('  Active rows propagated:', f'{len(orbit_audit):,}', '/', f'{len(active_omm):,}')
print('  Satellites entering 915-1065 km:', f'{len(in_window):,}')
print('  C-SWIM classified satellite match rate in window:', f'{satellite_match_rate:.1%}')
print('  Classified time-sample match rate in window:', f'{sample_match_rate:.1%}')
print('  Main-service counts:', in_window.loc[main_service_in_window].service.value_counts().to_dict())


## 2. Rebuild the audited ORDEM response library


In [ ]:
def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)


def read_ordem_table(path):
    lines = Path(path).read_text(errors='replace').splitlines()
    header_index = next(i for i, line in enumerate(lines) if line.lstrip().startswith('Element'))
    columns = lines[header_index].split()
    rows = []
    for line in lines[header_index + 1:]:
        fields = line.split()
        if len(fields) != len(columns):
            continue
        try:
            rows.append([float(value) for value in fields])
        except ValueError:
            continue
    frame = pd.DataFrame(rows, columns=columns)
    if frame.empty:
        raise ValueError(f'No numeric ORDEM table parsed from {path}')
    frame['Element'] = frame.Element.astype(int)
    return frame


def parse_elevation(path):
    for line in Path(path).read_text(errors='replace').splitlines():
        if '!' not in line:
            continue
        raw, comment = line.split('!', 1)
        if comment.strip().lower().startswith('elevation'):
            return float(raw.strip())
    raise ValueError(f'No elevation in {path}')


def overlap_fraction(low, high):
    low = np.asarray(low, dtype=float)
    high = np.asarray(high, dtype=float)
    overlap = np.maximum(0.0, np.minimum(high, RANGE_MAX_KM) - np.maximum(low, RANGE_MIN_KM))
    return np.divide(overlap, high - low, out=np.zeros_like(overlap), where=high > low)


old_response = pd.read_csv(OLD_RESPONSE_PATH)
old_centerline = old_response.loc[np.isclose(old_response.azimuth_deg, 90.0)].copy()
old_expected = np.arange(65.0, 85.0 + 0.01, 2.5)
if set(np.round(old_centerline.elevation_deg.unique(), 6)) != set(np.round(old_expected, 6)):
    raise RuntimeError('The old response table lacks the nine exact azimuth-90 elevations.')

response_by_size = {key: {} for key in SIZE_CASES}
relative_sd_by_size = {key: {} for key in SIZE_CASES}

for size_key, spec in SIZE_CASES.items():
    for elevation, group in old_centerline.groupby('elevation_deg'):
        mids = 0.5 * (group.alt_low.to_numpy() + group.alt_high.to_numpy())
        bins = np.searchsorted(ALTITUDE_EDGES, mids, side='right') - 1
        values = group[spec['old_column']].to_numpy(dtype=float)
        response_by_size[size_key][float(elevation)] = np.asarray([
            values[bins == index].sum() for index in range(N_SHELLS)
        ])

with tempfile.TemporaryDirectory(prefix='debris_closure_ordem_') as temporary:
    safe_extract(REFINEMENT_ZIP, temporary)
    result_dirs = sorted({path.parent for path in Path(temporary).rglob('ORDEM.IN')})
    if len(result_dirs) != 40:
        raise RuntimeError(f'Expected 40 b982 result directories; found {len(result_dirs)}')
    for directory in result_dirs:
        if 'Complete' not in (directory / 'status.txt').read_text(errors='replace'):
            raise RuntimeError(f'Incomplete ORDEM result: {directory.name}')
        elevation = parse_elevation(directory / 'ORDEM.IN')
        flux = read_ordem_table(directory / 'IGLOOFLUX_TEL.OUT')
        sigma_range = read_ordem_table(directory / 'IGLOOFLUX_SIGMARAN_TEL.OUT')
        sigma_population = read_ordem_table(directory / 'IGLOOFLUX_SIGMAPOP_TEL.OUT')
        overlap = overlap_fraction(flux.rng_low, flux.rng_high)
        mids = 0.5 * (flux.alt_low.to_numpy() + flux.alt_high.to_numpy())
        bins = np.searchsorted(ALTITUDE_EDGES, mids, side='right') - 1
        for size_key, spec in SIZE_CASES.items():
            flux_columns = [c for c in flux if c.startswith('Flux_') and c.endswith(spec['suffix'])]
            range_columns = [f"Sig(ran)_{c.split('_', 1)[1]}" for c in flux_columns]
            population_columns = [f"Sig(pop)_{c.split('_', 1)[1]}" for c in flux_columns]
            if len(flux_columns) != 5:
                raise RuntimeError(f'{directory.name}/{size_key}: expected five material populations')
            contribution = flux[flux_columns].sum(axis=1).to_numpy() * overlap
            row = np.asarray([contribution[bins == index].sum() for index in range(N_SHELLS)])
            response_by_size[size_key][float(elevation)] = row
            element_range_sd = np.sqrt(np.square(sigma_range[range_columns].to_numpy()).sum(axis=1)) * overlap
            element_population_sd = np.sqrt(np.square(sigma_population[population_columns].to_numpy()).sum(axis=1)) * overlap
            total_sd = float(np.sqrt(np.square(np.hypot(element_range_sd, element_population_sd)).sum()))
            relative_sd_by_size[size_key][float(elevation)] = total_sd / max(float(contribution.sum()), 1e-30)

H_BY_SIZE = {}
RESPONSE_LOG_SD_BY_SIZE = {}
tidy_response_rows = []
identifiability_rows = []
for size_key, spec in SIZE_CASES.items():
    available_elevations = np.asarray(list(response_by_size[size_key].keys()), dtype=float)
    if set(np.round(available_elevations, 6)) != set(np.round(ELEVATIONS, 6)):
        missing_elevations = sorted(set(np.round(ELEVATIONS, 6)) - set(np.round(available_elevations, 6)))
        raise RuntimeError(f'{size_key}: missing elevations {missing_elevations}')
    matrix = np.stack([response_by_size[size_key][float(elevation)] for elevation in ELEVATIONS])
    if not np.isfinite(matrix).all() or (matrix < 0).any() or np.any(matrix.sum(axis=1) <= 0):
        raise RuntimeError(f'Invalid response matrix for {size_key}')
    H_BY_SIZE[size_key] = matrix
    known_elevations = np.asarray(sorted(relative_sd_by_size[size_key]), dtype=float)
    known_sd = np.asarray([relative_sd_by_size[size_key][value] for value in known_elevations])
    interpolated_sd = np.interp(ELEVATIONS, known_elevations, known_sd)
    RESPONSE_LOG_SD_BY_SIZE[size_key] = np.sqrt(np.log1p(np.square(interpolated_sd)))
    normalized = matrix / matrix.sum(axis=1, keepdims=True)
    singular_values = np.linalg.svd(normalized, compute_uv=False)
    cumulative = np.cumsum(np.square(singular_values)) / np.square(singular_values).sum()
    effective_rank = float(np.square(np.square(singular_values).sum()) / np.power(singular_values, 4).sum())
    for component, (value, fraction) in enumerate(zip(singular_values, cumulative), start=1):
        identifiability_rows.append({
            'size_case': size_key, 'size_label': spec['label'], 'component': component,
            'singular_value': value, 'cumulative_energy_fraction': fraction,
            'effective_rank': effective_rank,
            'components_for_99_percent': int(np.searchsorted(cumulative, 0.99) + 1),
        })
    for action, elevation in enumerate(ELEVATIONS):
        for shell, altitude in enumerate(ALTITUDE_CENTERS):
            tidy_response_rows.append({
                'size_case': size_key, 'size_label': spec['label'],
                'elevation_deg': elevation, 'altitude_center_km': altitude,
                'response_m2_yr': matrix[action, shell],
                'response_share_within_action': normalized[action, shell],
                'response_log_sd': RESPONSE_LOG_SD_BY_SIZE[size_key][action],
            })

size_response = pd.DataFrame(tidy_response_rows)
identifiability = pd.DataFrame(identifiability_rows)

                                                                                
frozen_fine_response = pd.read_csv(FINE_RESPONSE_PATH)
frozen_shell_columns = [column for column in frozen_fine_response if column.startswith('shell_')]
frozen_h = frozen_fine_response[frozen_shell_columns].to_numpy(dtype=float)
if frozen_h.shape != (N_ACTIONS, N_SHELLS):
    raise RuntimeError(f'Frozen fine response shape changed: {frozen_h.shape}')
if not np.allclose(
    frozen_fine_response.elevation_deg.to_numpy(dtype=float), ELEVATIONS,
    atol=1e-10, rtol=0.0,
):
    raise RuntimeError('Frozen fine response elevations changed.')
relative_rebuild_error = np.max(
    np.abs(H_BY_SIZE['gt_1mm'] - frozen_h) / np.maximum(np.abs(frozen_h), 1e-30)
)
absolute_rebuild_error = np.max(np.abs(H_BY_SIZE['gt_1mm'] - frozen_h))
if not np.allclose(H_BY_SIZE['gt_1mm'], frozen_h, rtol=2e-7, atol=1e-12):
    raise RuntimeError(
        f'Rebuilt >1 mm response does not reproduce the frozen matrix: '
        f'max relative {relative_rebuild_error:.3e}, absolute {absolute_rebuild_error:.3e}'
    )

size_response.to_csv(OUTPUT_DIR / 'size_specific_response_library_49x10.csv.gz', index=False, compression='gzip')
identifiability.to_csv(OUTPUT_DIR / 'response_identifiability_by_size.csv', index=False)

print('SIZE RESPONSE AUDIT PASSED')
print('  Rebuilt >1 mm maximum relative difference:', f'{relative_rebuild_error:.3e}')
for size_key, spec in SIZE_CASES.items():
    group = identifiability.loc[identifiability.size_case == size_key].iloc[0]
    print(f"  {spec['label']}: {int(group.components_for_99_percent)} components for 99%; effective rank {group.effective_rank:.2f}")


## 3. Reuse the validated Bayesian inference model


In [ ]:
shell_distance = np.abs(ALTITUDE_CENTERS[:, None] - ALTITUDE_CENTERS[None, :])
scaled_distance = np.sqrt(3.0) * shell_distance / PRIOR_CORRELATION_LENGTH_KM
matern32 = (1.0 + scaled_distance) * np.exp(-scaled_distance)
K_X = PRIOR_LOG_SD**2 * (
    (1.0 - PRIOR_NUGGET_FRACTION) * matern32
    + PRIOR_NUGGET_FRACTION * np.eye(N_SHELLS)
)
K_X += np.eye(N_SHELLS) * 1e-8
EQUAL_WEIGHTS = np.repeat(1.0 / N_SHELLS, N_SHELLS)

eigenvalues, eigenvectors = np.linalg.eigh(K_X)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]
KL5_BASIS = eigenvectors[:, :5]
KL5_COVARIANCE = np.diag(eigenvalues[:5])
KL5_PRIOR_VARIANCE_FRACTION = float(eigenvalues[:5].sum() / eigenvalues.sum())


@dataclass
class InferenceModel:
    condition_name: str
    h: np.ndarray
    basis: np.ndarray
    latent_covariance: np.ndarray
    k_q: np.ndarray
    allowed_actions: np.ndarray
    signal_rate_center: float
    background_rate: float
    hidden_fraction: float
    side_lobe_fraction: float
    size_case: str
    basis_name: str
    prior_covariance: np.ndarray | None = None
    prior_cholesky: np.ndarray | None = None
    best_fixed_action: int | None = None
    fixed_design_variance: np.ndarray | None = None

    def __post_init__(self):
        m = self.basis.shape[1]
        prior = np.zeros((m + N_ACTIONS, m + N_ACTIONS))
        prior[:m, :m] = self.latent_covariance
        prior[m:, m:] = self.k_q
        self.prior_covariance = prior
        self.prior_cholesky = np.linalg.cholesky(prior)

    @property
    def n_modes(self):
        return self.basis.shape[1]

    @property
    def n_state(self):
        return self.n_modes + N_ACTIONS

    @property
    def response_correction(self):
        return 0.5 * np.diag(self.k_q)

    @property
    def signal_scale(self):
        return self.signal_rate_center * DWELL_HOURS / self.h[CENTER_ACTION].sum()

    @property
    def x_prior_covariance(self):
        return self.basis @ self.latent_covariance @ self.basis.T


def response_covariance(size_case, correlation_deg=1.5):
    action_distance = np.abs(ELEVATIONS[:, None] - ELEVATIONS[None, :])
    correlation = 0.90 * np.exp(-0.5 * np.square(action_distance / correlation_deg)) + 0.10 * np.eye(N_ACTIONS)
    sd = RESPONSE_LOG_SD_BY_SIZE[size_case]
    covariance = sd[:, None] * correlation * sd[None, :]
    covariance += np.eye(N_ACTIONS) * 1e-8
    return covariance


def make_model(spec):
    size_case = spec.get('size_case', 'gt_1mm')
    basis_name = spec.get('basis', 'full10')
    if basis_name == 'full10':
        basis, latent_covariance = np.eye(N_SHELLS), K_X
    elif basis_name == 'kl5':
        basis, latent_covariance = KL5_BASIS, KL5_COVARIANCE
    else:
        raise ValueError(basis_name)
    hidden_fraction = float(spec.get('hidden_fraction', 1.0))
    signal_rate = hidden_fraction * NET_UNRESOLVED_SIGNAL_RATE
    background_rate = FALSE_RATE_PER_HOUR + (1.0 - hidden_fraction) * NET_UNRESOLVED_SIGNAL_RATE
    domain = spec.get('action_domain', '65_89')
    if domain == '65_89':
        allowed = np.arange(N_ACTIONS)
    elif domain == '70_85':
        allowed = np.where((ELEVATIONS >= 70.0) & (ELEVATIONS <= 85.0))[0]
    else:
        raise ValueError(domain)
    return InferenceModel(
        condition_name=spec['condition'], h=H_BY_SIZE[size_case].copy(), basis=basis,
        latent_covariance=latent_covariance.copy(), k_q=response_covariance(size_case),
        allowed_actions=allowed, signal_rate_center=signal_rate,
        background_rate=background_rate, hidden_fraction=hidden_fraction,
        side_lobe_fraction=float(spec.get('side_lobe_fraction', 0.0)),
        size_case=size_case, basis_name=basis_name,
    )


def state_x(model, state):
    return model.basis @ state[:model.n_modes]


def theta_moments(model, mean, covariance):
    x_mean = model.basis @ mean[:model.n_modes]
    x_covariance = model.basis @ covariance[:model.n_modes, :model.n_modes] @ model.basis.T
    diagonal = np.maximum(np.diag(x_covariance), 0.0)
    theta_mean = np.exp(x_mean + 0.5 * diagonal)
    theta_variance = (np.exp(diagonal) - 1.0) * np.exp(2.0 * x_mean + diagonal)
    return x_mean, x_covariance, theta_mean, theta_variance


def observation_linearization(model, mean, covariance, action):
    x = state_x(model, mean)
    q = mean[model.n_modes + action]
    components = (
        model.signal_scale * np.exp(q - model.response_correction[action])
        * model.h[action] * np.exp(x)
    )
    signal = float(components.sum())
    expected = model.background_rate * DWELL_HOURS + signal
    gradient = np.zeros(model.n_state)
    gradient[:model.n_modes] = model.basis.T @ components
    gradient[model.n_modes + action] = signal
    variance = expected + expected**2 / INFERENCE_DISPERSION
    return expected, gradient, variance


def sequential_update(model, mean, covariance, action, observed, iterations=5):
    prior_mean = mean.copy()
    prior_covariance = covariance.copy()
    trial = prior_mean.copy()
    for _ in range(iterations):
        expected, gradient, observation_variance = observation_linearization(model, trial, prior_covariance, action)
        projected = prior_covariance @ gradient
        denominator = observation_variance + gradient @ projected
        adjusted = observed - expected + gradient @ (trial - prior_mean)
        proposal = prior_mean + projected * adjusted / max(denominator, 1e-10)
        trial = 0.35 * trial + 0.65 * proposal
    expected, gradient, observation_variance = observation_linearization(model, trial, prior_covariance, action)
    projected = prior_covariance @ gradient
    updated = prior_covariance - np.outer(projected, projected) / max(
        observation_variance + gradient @ projected, 1e-10
    )
    updated = 0.5 * (updated + updated.T)
    values, vectors = np.linalg.eigh(updated)
    updated = (vectors * np.maximum(values, 1e-10)) @ vectors.T
    return trial, updated


def batch_map_laplace(model, actions, counts, start_state):
    actions = np.asarray(actions, dtype=int)
    counts = np.asarray(counts, dtype=float)
    observed_h = model.h[actions]
    corrections = model.response_correction[actions]
    cholesky = model.prior_cholesky

    def objective_gradient(whitened):
        state = cholesky @ whitened
        x = state_x(model, state)
        q = state[model.n_modes:][actions]
        components = (
            model.signal_scale * np.exp(q - corrections)[:, None]
            * observed_h * np.exp(x)[None, :]
        )
        signal = components.sum(axis=1)
        means = np.maximum(model.background_rate + signal, 1e-12)
        k = INFERENCE_DISPERSION
        log_likelihood = (
            gammaln(counts + k) - gammaln(k) - gammaln(counts + 1.0)
            + k * np.log(k) + counts * np.log(means) - (counts + k) * np.log(means + k)
        )
        derivative_nll_mean = k * (means - counts) / (means * (means + k))
        gradient_state = np.zeros(model.n_state)
        gradient_state[:model.n_modes] = model.basis.T @ (
            derivative_nll_mean[:, None] * components
        ).sum(axis=0)
        np.add.at(gradient_state[model.n_modes:], actions, derivative_nll_mean * signal)
        return float(0.5 * whitened @ whitened - log_likelihood.sum()), whitened + cholesky.T @ gradient_state

    initial = np.linalg.solve(cholesky, np.asarray(start_state, dtype=float))
    result = minimize(
        objective_gradient, initial, jac=True, method='L-BFGS-B',
        bounds=[(-8.0, 8.0)] * model.n_state,
        options={'maxiter': 200, 'maxls': 40, 'ftol': 1e-10, 'gtol': 2e-6},
    )
    state = cholesky @ result.x
    x = state_x(model, state)
    q = state[model.n_modes:][actions]
    components = (
        model.signal_scale * np.exp(q - corrections)[:, None]
        * observed_h * np.exp(x)[None, :]
    )
    signal = components.sum(axis=1)
    means = np.maximum(model.background_rate + signal, 1e-12)
    gradient_state = np.zeros((len(actions), model.n_state))
    gradient_state[:, :model.n_modes] = components @ model.basis
    gradient_state[np.arange(len(actions)), model.n_modes + actions] = signal
    gradient_white = gradient_state @ cholesky
    observation_variance = means + means**2 / INFERENCE_DISPERSION
    precision_white = np.eye(model.n_state) + gradient_white.T @ (
        gradient_white / observation_variance[:, None]
    )
    covariance = cholesky @ np.linalg.inv(precision_white) @ cholesky.T
    covariance = 0.5 * (covariance + covariance.T)
    gradient_norm = float(np.linalg.norm(result.jac, ord=np.inf))
    success = bool(result.success or (np.isfinite(result.fun) and gradient_norm < 1e-4))
    return state, covariance, success, int(result.nit), gradient_norm


def posterior_metrics(model, mean, covariance, truth_x):
    x_mean, x_covariance, theta_mean, theta_variance = theta_moments(model, mean, covariance)
    truth_theta = np.exp(truth_x)
    x_sd = np.sqrt(np.maximum(np.diag(x_covariance), 0.0))
    low95 = np.exp(x_mean - 1.96 * x_sd)
    high95 = np.exp(x_mean + 1.96 * x_sd)
    prior_mean = np.zeros(model.n_state)
    _, _, _, initial_theta_variance = theta_moments(model, prior_mean, model.prior_covariance)
    return {
        'equal_uncertainty_fraction': float(
            np.sum(EQUAL_WEIGHTS * theta_variance) / np.sum(EQUAL_WEIGHTS * initial_theta_variance)
        ),
        'equal_weighted_rmse': float(np.sqrt(np.sum(EQUAL_WEIGHTS * np.square(theta_mean - truth_theta)))),
        'cellwise_95_coverage': float(np.mean((truth_theta >= low95) & (truth_theta <= high95))),
        'aggregate_log_mean': float(EQUAL_WEIGHTS @ x_mean),
        'aggregate_log_sd': float(np.sqrt(max(EQUAL_WEIGHTS @ x_covariance @ EQUAL_WEIGHTS, 1e-12))),
    }


def aopt_scores(model, mean, covariance):
    _, _, _, before = theta_moments(model, mean, covariance)
    x_mean, _, _, _ = theta_moments(model, mean, covariance)
    scores = np.full(N_ACTIONS, -np.inf)
    for action in model.allowed_actions:
        _, gradient, observation_variance = observation_linearization(model, mean, covariance, action)
        projected = covariance @ gradient
        updated = covariance - np.outer(projected, projected) / max(
            observation_variance + gradient @ projected, 1e-10
        )
        x_covariance = model.basis @ updated[:model.n_modes, :model.n_modes] @ model.basis.T
        diagonal = np.maximum(np.diag(x_covariance), 0.0)
        after = (np.exp(diagonal) - 1.0) * np.exp(2.0 * x_mean + diagonal)
        scores[action] = np.sum(EQUAL_WEIGHTS * (before - after))
    return scores


def eig_scores(model, mean, covariance):
    scores = np.full(N_ACTIONS, -np.inf)
    for action in model.allowed_actions:
        _, gradient, observation_variance = observation_linearization(model, mean, covariance, action)
        ratio = gradient @ covariance @ gradient / observation_variance
        scores[action] = 0.5 * np.log1p(max(ratio, 0.0))
    return scores


def select_best_fixed(model):
    predicted = np.full(N_ACTIONS, np.inf)
    for action in model.allowed_actions:
        mean = np.zeros(model.n_state)
        covariance = model.prior_covariance.copy()
        for _ in range(BUDGET):
            _, gradient, observation_variance = observation_linearization(model, mean, covariance, action)
            projected = covariance @ gradient
            covariance -= np.outer(projected, projected) / max(
                observation_variance + gradient @ projected, 1e-10
            )
            covariance = 0.5 * (covariance + covariance.T)
        predicted[action] = posterior_metrics(model, mean, covariance, np.zeros(N_SHELLS))['equal_uncertainty_fraction']
    model.best_fixed_action = int(np.argmin(predicted))
    model.fixed_design_variance = predicted


def choose_action(model, policy, mean, covariance, rng):
    if policy == 'best_fixed_equal':
        return model.best_fixed_action
    if policy == 'random':
        return int(rng.choice(model.allowed_actions))
    if policy == 'unweighted_aopt':
        return int(np.argmax(aopt_scores(model, mean, covariance)))
    if policy == 'eig':
        return int(np.argmax(eig_scores(model, mean, covariance)))
    raise ValueError(policy)

print('MODEL CODE READY')
print('  Five KL modes capture', f'{100 * KL5_PRIOR_VARIANCE_FRACTION:.1f}%', 'of prior log-field variance')


## 4. Freeze the service target before looking at policy results


In [ ]:
def normalize(vector):
    vector = np.asarray(vector, dtype=float)
    total = vector.sum()
    if total <= 0 or not np.isfinite(total):
        raise ValueError('Cannot normalize an empty or invalid target vector.')
    return vector / total

service_totals = category_shell_samples.sum(axis=1)
present_mask = service_totals > 0
present_services = [s for s, present in zip(SERVICE_CATEGORIES, present_mask) if present]
if len(present_services) < 3:
    raise RuntimeError(f'Only {len(present_services)} C-SWIM service categories enter the altitude window.')

service_profiles = np.vstack([
    normalize(row) if row.sum() > 0 else np.zeros(N_SHELLS)
    for row in category_shell_samples
])
service_value_profiles = np.vstack([
    normalize(row) if row.sum() > 0 else np.zeros(N_SHELLS)
    for row in category_shell_value
])

                                                                                              
                                                                      
PRIMARY_SERVICE_WEIGHTS = normalize(service_profiles[present_mask].mean(axis=0))
VALUE_WITHIN_SERVICE_WEIGHTS = normalize(service_value_profiles[present_mask].mean(axis=0))
ALL_ACTIVE_WEIGHTS = normalize(all_shell_samples)
UNCLASSIFIED_PROFILE = normalize(unclassified_shell_samples) if unclassified_shell_samples.sum() > 0 else None
INCLUDING_UNCLASSIFIED_WEIGHTS = normalize(
    np.vstack([
        service_profiles[present_mask],
        *([] if UNCLASSIFIED_PROFILE is None else [UNCLASSIFIED_PROFILE]),
    ]).mean(axis=0)
)

TARGET_SPECS = [
    {
        'target_name': 'primary_equal_service',
        'display_label': 'Equal C-SWIM services (primary)',
        'weights': PRIMARY_SERVICE_WEIGHTS,
        'analysis_role': 'service_extension_primary',
        'n_replicates': N_PRIMARY,
        'size_case': 'gt_1cm',
        'truth_scenario': 'model_matched',
    },
    {
        'target_name': 'replacement_value_within_service',
        'display_label': 'C-SWIM value within each service',
        'weights': VALUE_WITHIN_SERVICE_WEIGHTS,
        'analysis_role': 'weight_sensitivity',
        'n_replicates': N_SENSITIVITY,
        'size_case': 'gt_1cm',
        'truth_scenario': 'model_matched',
    },
    {
        'target_name': 'all_active_equal_object',
        'display_label': 'All active satellites, equal object',
        'weights': ALL_ACTIVE_WEIGHTS,
        'analysis_role': 'weight_sensitivity',
        'n_replicates': N_SENSITIVITY,
        'size_case': 'gt_1cm',
        'truth_scenario': 'model_matched',
    },
    {
        'target_name': 'services_plus_unclassified_group',
        'display_label': 'Services plus unclassified as one group',
        'weights': INCLUDING_UNCLASSIFIED_WEIGHTS,
        'analysis_role': 'classification_sensitivity',
        'n_replicates': N_SENSITIVITY,
        'size_case': 'gt_1cm',
        'truth_scenario': 'model_matched',
    },
]

for service_index_value, service in enumerate(SERVICE_CATEGORIES):
    if not present_mask[service_index_value] or len(present_services) <= 3:
        continue
    kept = [i for i, present in enumerate(present_mask) if present and i != service_index_value]
    TARGET_SPECS.append({
        'target_name': f'leave_out_{service.lower()}',
        'display_label': f'Leave out {service}',
        'weights': normalize(service_profiles[kept].mean(axis=0)),
        'analysis_role': 'leave_one_service_out',
        'n_replicates': N_SENSITIVITY,
        'size_case': 'gt_1cm',
        'truth_scenario': 'model_matched',
    })

TARGET_SPECS.extend([
    {
        'target_name': 'primary_localized_anomaly',
        'display_label': 'Primary target, localized anomaly',
        'weights': PRIMARY_SERVICE_WEIGHTS,
        'analysis_role': 'population_misspecification',
        'n_replicates': N_SENSITIVITY,
        'size_case': 'gt_1cm',
        'truth_scenario': 'localized_anomaly',
    },
    {
        'target_name': 'primary_neighbor_mixing',
        'display_label': 'Primary target, neighboring-shell mixing',
        'weights': PRIMARY_SERVICE_WEIGHTS,
        'analysis_role': 'population_misspecification',
        'n_replicates': N_SENSITIVITY,
        'size_case': 'gt_1cm',
        'truth_scenario': 'neighbor_mixing',
    },
    {
        'target_name': 'primary_gt1mm_response',
        'display_label': 'Primary target, ORDEM >1 mm response',
        'weights': PRIMARY_SERVICE_WEIGHTS,
        'analysis_role': 'size_sensitivity',
        'n_replicates': N_SENSITIVITY,
        'size_case': 'gt_1mm',
        'truth_scenario': 'model_matched',
    },
])

extension_primary_spec = {
    'frozen_before_policy_simulation': True,
    'classification_source': 'C-SWIM v1, DOI 10.5281/zenodo.19477756',
    'classification_rule': 'exact C-SWIM NAV/COMMS/WEATHER/EO/MIL mapping; no subjective ranking',
    'primary_target': 'equal total influence for each represented C-SWIM service; equal satellites within service',
    'present_services': present_services,
    'absent_services': [s for s in SERVICE_CATEGORIES if s not in present_services],
    'unclassified_handling': 'excluded from primary service target; included as a separate group in a named sensitivity',
    'primary_size_response': (
        'ORDEM >1 cm; selected before this extension run because the published 2018 EISCAT study '
        'describes Tromso and Svalbard as sensitive to objects down to a few centimeters'
    ),
    'size_sensitivity': 'ORDEM >1 mm retained as a named sensitivity; the radar is not claimed to detect all 1 mm objects',
    'primary_question': (
        'Does service-aware A-optimal scheduling reduce the C-SWIM equal-service posterior variance '
        'more than the identical unweighted adaptive scheduler after 24 one-hour observations?'
    ),
    'primary_policy': 'target_aopt',
    'primary_comparator': 'unweighted_aopt',
    'primary_endpoint': 'C-SWIM equal-service weighted posterior variance fraction after 24 observations',
    'primary_success_rule': 'paired bootstrap 95% CI lower bound for reduction versus unweighted A-opt is greater than 0%',
    'secondary_comparators': ['target-matched best fixed', 'random', 'expected information gain'],
    'multiplicity_rule': 'only target_aopt versus unweighted_aopt on primary_equal_service is confirmatory',
    'radar_role': '2018 Tromso data calibrate detector/count model; alternative pointings are prospective ORDEM simulations',
    'asset_role': '2026 cached active-satellite snapshot defines prospective service exposure',
    'not_claimed': ['individual collision warning', 'inclination-resolved debris map', '2018 service exposure reconstruction'],
    'n_primary_worlds': N_PRIMARY,
    'n_sensitivity_worlds': N_SENSITIVITY,
    'master_seed': MASTER_SEED,
}

input_hashes = {
    path.name: sha256_file(path) for path in [
        REFINEMENT_ZIP, OLD_RESPONSE_PATH, FINE_RESPONSE_PATH, EVENT_PATH,
        SELECTED_DETECTOR_PATH, PREVIOUS_CONFIG_PATH, SNAPSHOT_PATH,
        CSWIM_CLASSIFIED_PATH, CSWIM_COST_PATH, CSWIM_SOURCE_PATH,
    ]
}
config_payload = {
    'notebook_version': '2026-08-31.2',
    'primary_spec': extension_primary_spec,
    'targets': [
        {k: v for k, v in spec.items() if k != 'weights'}
        | {'weights': np.asarray(spec['weights']).round(15).tolist()}
        for spec in TARGET_SPECS
    ],
    'input_hashes': input_hashes,
    'budget': BUDGET,
}
config_hash = hashlib.sha256(json.dumps(config_payload, sort_keys=True).encode()).hexdigest()[:12]
OUTPUT_DIR = PROJECT_ROOT / 'phase3_outputs' / 'service_weighted_validation' / f'cswim_service_{config_hash}'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

orbit_audit.to_csv(OUTPUT_DIR / 'active_satellite_cswim_classification_audit.csv.gz', index=False, compression='gzip')
failure_audit.to_csv(OUTPUT_DIR / 'active_satellite_cswim_propagation_failures.csv', index=False)

weight_rows = []
for shell, center in enumerate(ALTITUDE_CENTERS):
    row = {
        'shell_index': shell,
        'altitude_center_km': center,
        'primary_equal_service_weight': PRIMARY_SERVICE_WEIGHTS[shell],
        'all_active_equal_object_weight': ALL_ACTIVE_WEIGHTS[shell],
        'value_within_service_weight': VALUE_WITHIN_SERVICE_WEIGHTS[shell],
        'unclassified_samples': unclassified_shell_samples[shell],
        'all_active_samples': all_shell_samples[shell],
    }
    for i, service in enumerate(SERVICE_CATEGORIES):
        row[f'{service.lower()}_samples'] = category_shell_samples[i, shell]
        row[f'{service.lower()}_profile'] = service_profiles[i, shell]
    weight_rows.append(row)
service_weight_table = pd.DataFrame(weight_rows)
service_weight_table.to_csv(OUTPUT_DIR / 'cswim_service_shell_weights.csv', index=False)

classification_summary = {
    'active_snapshot_rows': int(len(active_omm)),
    'propagated_rows': int(len(orbit_audit)),
    'satellites_entering_window': int(len(in_window)),
    'satellite_classification_match_rate_in_window': satellite_match_rate,
    'time_sample_classification_match_rate_in_window': sample_match_rate,
    'present_services': present_services,
    'absent_services': [s for s in SERVICE_CATEGORIES if s not in present_services],
    'main_service_satellite_counts_in_window': {
        str(k): int(v) for k, v in in_window.loc[main_service_in_window].service.value_counts().items()
    },
    'unclassified_satellites_in_window': int((in_window.service == 'UNCLASSIFIED').sum()),
    'other_satellites_in_window': int((in_window.service == 'OTHER').sum()),
    'mapping_count_check': reproduced_counts,
}
(OUTPUT_DIR / 'cswim_classification_summary.json').write_text(json.dumps(classification_summary, indent=2))
(OUTPUT_DIR / 'service_extension_primary_spec.json').write_text(json.dumps(extension_primary_spec, indent=2))
(OUTPUT_DIR / 'service_extension_config.json').write_text(json.dumps(config_payload, indent=2))

print('SERVICE TARGET FROZEN BEFORE POLICY RESULTS')
print('  Output:', OUTPUT_DIR)
print('  Present services:', present_services)
print('  Absent services:', extension_primary_spec['absent_services'])
print('  Classification match rate:', f'{satellite_match_rate:.1%}')
display(service_weight_table)


## 5. Paired service-aware policy experiment


In [ ]:
POLICIES = ('best_fixed_target', 'random', 'unweighted_aopt', 'target_aopt', 'eig')
POLICY_LABELS = {
    'best_fixed_target': 'Target-matched best fixed',
    'random': 'Random',
    'unweighted_aopt': 'Unweighted adaptive',
    'target_aopt': 'C-SWIM service-aware adaptive',
    'eig': 'Adaptive information gain',
}
POLICY_COLORS = {
    'best_fixed_target': '#595959',
    'random': '#4C78A8',
    'unweighted_aopt': '#F2A541',
    'target_aopt': '#E45756',
    'eig': '#54A24B',
}

def target_metrics(model, mean, covariance, truth_x, weights):
    weights = normalize(weights)
    x_mean, x_covariance, theta_mean, theta_variance = theta_moments(model, mean, covariance)
    truth_theta = np.exp(truth_x)
    x_sd = np.sqrt(np.maximum(np.diag(x_covariance), 0.0))
    low95 = np.exp(x_mean - 1.96 * x_sd)
    high95 = np.exp(x_mean + 1.96 * x_sd)
    prior_mean = np.zeros(model.n_state)
    _, _, _, initial_theta_variance = theta_moments(model, prior_mean, model.prior_covariance)
    result = {
        'target_uncertainty_fraction': float(np.sum(weights * theta_variance) / np.sum(weights * initial_theta_variance)),
        'target_weighted_rmse': float(np.sqrt(np.sum(weights * np.square(theta_mean - truth_theta)))),
        'cellwise_95_coverage': float(np.mean((truth_theta >= low95) & (truth_theta <= high95))),
    }
    for i, service in enumerate(SERVICE_CATEGORIES):
        if service_profiles[i].sum() > 0:
            sw = normalize(service_profiles[i])
            result[f'{service.lower()}_uncertainty_fraction'] = float(
                np.sum(sw * theta_variance) / np.sum(sw * initial_theta_variance)
            )
        else:
            result[f'{service.lower()}_uncertainty_fraction'] = np.nan
    return result

def weighted_aopt_scores(model, mean, covariance, weights):
    weights = normalize(weights)
    _, _, _, before = theta_moments(model, mean, covariance)
    x_mean, _, _, _ = theta_moments(model, mean, covariance)
    scores = np.full(N_ACTIONS, -np.inf)
    for action in model.allowed_actions:
        _, gradient, observation_variance = observation_linearization(model, mean, covariance, action)
        projected = covariance @ gradient
        updated = covariance - np.outer(projected, projected) / max(
            observation_variance + gradient @ projected, 1e-10
        )
        x_covariance = model.basis @ updated[:model.n_modes, :model.n_modes] @ model.basis.T
        diagonal = np.maximum(np.diag(x_covariance), 0.0)
        after = (np.exp(diagonal) - 1.0) * np.exp(2.0 * x_mean + diagonal)
        scores[action] = np.sum(weights * (before - after))
    return scores

def target_best_fixed(model, weights):
    predicted = np.full(N_ACTIONS, np.inf)
    for action in model.allowed_actions:
        mean = np.zeros(model.n_state)
        covariance = model.prior_covariance.copy()
        for _ in range(BUDGET):
            _, gradient, observation_variance = observation_linearization(model, mean, covariance, action)
            projected = covariance @ gradient
            covariance -= np.outer(projected, projected) / max(
                observation_variance + gradient @ projected, 1e-10
            )
            covariance = 0.5 * (covariance + covariance.T)
        predicted[action] = target_metrics(model, mean, covariance, np.zeros(N_SHELLS), weights)['target_uncertainty_fraction']
    return int(np.argmin(predicted)), predicted

def neighbor_mixing_matrix(fraction=0.15):
    matrix = np.zeros((N_SHELLS, N_SHELLS))
    for source in range(N_SHELLS):
        neighbors = []
        if source > 0:
            neighbors.append(source - 1)
        if source < N_SHELLS - 1:
            neighbors.append(source + 1)
        matrix[source, source] = 1.0 - fraction
        for destination in neighbors:
            matrix[destination, source] += fraction / len(neighbors)
    return matrix

MIXING_MATRIX = neighbor_mixing_matrix(0.15)

def truth_streams(model, replicate, truth_scenario):
                                                                                                     
    rng = np.random.default_rng(stable_seed(MASTER_SEED, 'service_truth', truth_scenario, model.size_case, replicate))
    truth_x = rng.multivariate_normal(np.zeros(N_SHELLS), K_X)
    truth_q = rng.multivariate_normal(np.zeros(N_ACTIONS), model.k_q)
    if truth_scenario == 'localized_anomaly':
        center = int(rng.integers(N_SHELLS))
        width = float(rng.uniform(8.0, 15.0))
        amplitude = float(rng.uniform(np.log(2.0), np.log(3.0)))
        truth_x += amplitude * np.exp(-0.5 * np.square((ALTITUDE_CENTERS - ALTITUDE_CENTERS[center]) / width))
    elif truth_scenario == 'neighbor_mixing':
        truth_x = np.log(np.maximum(MIXING_MATRIX @ np.exp(truth_x), 1e-12))
    elif truth_scenario != 'model_matched':
        raise ValueError(truth_scenario)
    truth_h = model.h * np.exp(truth_q[:, None] - model.response_correction[:, None])
    means = model.background_rate + model.signal_scale * (truth_h @ np.exp(truth_x))
    count_rng = np.random.default_rng(stable_seed(MASTER_SEED, 'service_counts', truth_scenario, model.size_case, replicate))
    probability = INFERENCE_DISPERSION / (INFERENCE_DISPERSION + means[:, None])
    streams = count_rng.negative_binomial(INFERENCE_DISPERSION, probability, size=(N_ACTIONS, BUDGET))
    return truth_x, streams

def choose_target_action(model, policy, mean, covariance, weights, best_fixed, rng):
    if policy == 'best_fixed_target':
        return best_fixed
    if policy == 'random':
        return int(rng.choice(model.allowed_actions))
    if policy == 'unweighted_aopt':
        return int(np.argmax(aopt_scores(model, mean, covariance)))
    if policy == 'target_aopt':
        return int(np.argmax(weighted_aopt_scores(model, mean, covariance, weights)))
    if policy == 'eig':
        return int(np.argmax(eig_scores(model, mean, covariance)))
    raise ValueError(policy)

def run_target_policy(target, model, policy, replicate, truth_x, streams, best_fixed):
                                                                                          
    rng = np.random.default_rng(stable_seed(MASTER_SEED, 'service_policy', policy, replicate))
    mean = np.zeros(model.n_state)
    covariance = model.prior_covariance.copy()
    visits = np.zeros(N_ACTIONS, dtype=int)
    history_actions, history_counts = [], []
    step_rows, action_rows = [], []
    base = {
        'target_name': target['target_name'], 'display_label': target['display_label'],
        'analysis_role': target['analysis_role'], 'size_case': target['size_case'],
        'truth_scenario': target['truth_scenario'], 'replicate': replicate, 'policy': policy,
    }
    step_rows.append(base | {'step': 0, **target_metrics(model, mean, covariance, truth_x, target['weights']), 'posterior_method': 'prior'})
    for step in range(1, BUDGET + 1):
        action = choose_target_action(model, policy, mean, covariance, target['weights'], best_fixed, rng)
        visit = int(visits[action])
        observed = int(streams[action, visit])
        visits[action] += 1
        predicted = observation_linearization(model, mean, covariance, action)[0]
        history_actions.append(action)
        history_counts.append(observed)
        mean, covariance = sequential_update(model, mean, covariance, action, observed)
        step_rows.append(base | {
            'step': step, **target_metrics(model, mean, covariance, truth_x, target['weights']),
            'posterior_method': 'sequential', 'cumulative_events': int(sum(history_counts)),
        })
        action_rows.append(base | {
            'step': step, 'action_index': action, 'elevation_deg': ELEVATIONS[action],
            'observed_events': observed, 'predicted_events_before_update': predicted,
            'visit_number': visit + 1,
        })
    final_mean, final_covariance, success, iterations, gradient_norm = batch_map_laplace(
        model, history_actions, history_counts, mean
    )
    step_rows[-1].update(target_metrics(model, final_mean, final_covariance, truth_x, target['weights']))
    step_rows[-1].update({
        'posterior_method': 'batch_map_fisher_laplace', 'optimizer_success': success,
        'optimizer_iterations': iterations, 'optimizer_gradient_inf_norm': gradient_norm,
    })
    return step_rows, action_rows

all_steps, all_actions = [], []
simulation_start = time.time()
for target_number, target in enumerate(TARGET_SPECS, start=1):
    model_spec = {
        'condition': target['target_name'], 'size_case': target['size_case'],
        'hidden_fraction': 1.0, 'side_lobe_fraction': 0.0,
        'basis': 'full10', 'action_domain': '65_89',
    }
    model = make_model(model_spec)
    best_fixed, fixed_curve = target_best_fixed(model, target['weights'])
    target['best_fixed_action'] = int(best_fixed)
    target['best_fixed_elevation_deg'] = float(ELEVATIONS[best_fixed])
    n_replicates = int(target['n_replicates'])
    step_path = CHECKPOINT_DIR / f"steps_{target['target_name']}.csv.gz"
    action_path = CHECKPOINT_DIR / f"actions_{target['target_name']}.csv.gz"
    if FORCE_RECOMPUTE:
        for path in (step_path, action_path):
            if path.exists():
                path.unlink()
    existing_steps = pd.read_csv(step_path) if step_path.exists() else pd.DataFrame()
    existing_actions = pd.read_csv(action_path) if action_path.exists() else pd.DataFrame()
    if len(existing_actions):
        counts = existing_actions.groupby(['replicate', 'policy']).size()
        complete = set(
            counts[counts == BUDGET].reset_index().groupby('replicate')
            .filter(lambda frame: set(frame.policy) == set(POLICIES)).replicate.astype(int)
        )
    else:
        complete = set()
    print(
        f"\nTarget {target_number}/{len(TARGET_SPECS)}: {target['display_label']}\n"
        f"  target-matched best fixed = {ELEVATIONS[best_fixed]:.1f} deg; resume {len(complete)}/{n_replicates}"
    )
    pending_steps, pending_actions = [], []
    for replicate in range(n_replicates):
        if replicate in complete:
            continue
        truth_x, streams = truth_streams(model, replicate, target['truth_scenario'])
        for policy in POLICIES:
            steps, actions = run_target_policy(target, model, policy, replicate, truth_x, streams, best_fixed)
            pending_steps.extend(steps)
            pending_actions.extend(actions)
        if (replicate + 1) % CHECKPOINT_EVERY == 0:
            existing_steps = pd.concat([existing_steps, pd.DataFrame(pending_steps)], ignore_index=True)
            existing_actions = pd.concat([existing_actions, pd.DataFrame(pending_actions)], ignore_index=True)
            existing_steps.to_csv(step_path, index=False, compression='gzip')
            existing_actions.to_csv(action_path, index=False, compression='gzip')
            pending_steps.clear(); pending_actions.clear()
            print(f'  checkpoint through world {replicate + 1}/{n_replicates}', flush=True)
    if pending_steps:
        existing_steps = pd.concat([existing_steps, pd.DataFrame(pending_steps)], ignore_index=True)
        existing_actions = pd.concat([existing_actions, pd.DataFrame(pending_actions)], ignore_index=True)
        existing_steps.to_csv(step_path, index=False, compression='gzip')
        existing_actions.to_csv(action_path, index=False, compression='gzip')
    expected_actions = n_replicates * len(POLICIES) * BUDGET
    if len(existing_actions) != expected_actions:
        raise RuntimeError(f"{target['target_name']}: {len(existing_actions)} actions; expected {expected_actions}")
    all_steps.append(existing_steps)
    all_actions.append(existing_actions)

step_metrics = pd.concat(all_steps, ignore_index=True)
action_history = pd.concat(all_actions, ignore_index=True)
step_metrics.to_csv(OUTPUT_DIR / 'service_extension_step_metrics.csv.gz', index=False, compression='gzip')
action_history.to_csv(OUTPUT_DIR / 'service_extension_action_history.csv.gz', index=False, compression='gzip')
print('\nSERVICE-EXTENSION SIMULATION COMPLETE')
print('  Minutes:', round((time.time() - simulation_start) / 60, 2))


## 6. One confirmatory extension test and labeled sensitivities


In [ ]:
final_metrics = step_metrics.loc[step_metrics.step == BUDGET].copy()

def paired_bootstrap(candidate, baseline, seed, draws=BOOTSTRAP_DRAWS):
    candidate = np.asarray(candidate, float)
    baseline = np.asarray(baseline, float)
    if len(candidate) != len(baseline):
        raise ValueError('Paired arrays differ in length.')
    rng = np.random.default_rng(seed)
    improvements = np.empty(draws)
    rmse_differences = np.empty(draws)
    n = len(candidate)
                                                                                           
    for draw in range(draws):
        idx = rng.integers(0, n, n)
        improvements[draw] = 100.0 * (1.0 - candidate[idx].mean() / baseline[idx].mean())
    point = 100.0 * (1.0 - candidate.mean() / baseline.mean())
    low, high = np.quantile(improvements, [0.025, 0.975])
    return float(point), float(low), float(high)

comparison_rows = []
for target_name, frame in final_metrics.groupby('target_name'):
    target_policy = frame.loc[frame.policy == 'target_aopt'].sort_values('replicate')
    for baseline_policy in ('unweighted_aopt', 'best_fixed_target', 'random', 'eig'):
        baseline = frame.loc[frame.policy == baseline_policy].sort_values('replicate')
        merged = target_policy.merge(baseline, on='replicate', suffixes=('_target', '_baseline'), validate='one_to_one')
        point, low, high = paired_bootstrap(
            merged.target_uncertainty_fraction_target,
            merged.target_uncertainty_fraction_baseline,
            stable_seed(MASTER_SEED, 'bootstrap', target_name, baseline_policy),
        )
        rmse_diff = merged.target_weighted_rmse_target - merged.target_weighted_rmse_baseline
        rng = np.random.default_rng(stable_seed(MASTER_SEED, 'rmse_bootstrap', target_name, baseline_policy))
        rmse_draws = []
        for _ in range(BOOTSTRAP_DRAWS):
            idx = rng.integers(0, len(merged), len(merged))
            rmse_draws.append(float(rmse_diff.iloc[idx].mean()))
        rmse_low, rmse_high = np.quantile(rmse_draws, [0.025, 0.975])
        meta = frame.iloc[0]
        comparison_rows.append({
            'target_name': target_name, 'display_label': meta.display_label,
            'analysis_role': meta.analysis_role, 'baseline': baseline_policy,
            'n_worlds': len(merged), 'uncertainty_improvement_percent': point,
            'uncertainty_improvement_ci95_low': low,
            'uncertainty_improvement_ci95_high': high,
            'target_rmse_minus_baseline': float(rmse_diff.mean()),
            'target_rmse_difference_ci95_low': float(rmse_low),
            'target_rmse_difference_ci95_high': float(rmse_high),
        })

paired_comparisons = pd.DataFrame(comparison_rows)
paired_comparisons.to_csv(OUTPUT_DIR / 'service_extension_paired_comparisons.csv', index=False)

final_summary = final_metrics.groupby(
    ['target_name', 'display_label', 'analysis_role', 'policy'], as_index=False
).agg(
    uncertainty_mean=('target_uncertainty_fraction', 'mean'),
    uncertainty_sd=('target_uncertainty_fraction', 'std'),
    rmse_mean=('target_weighted_rmse', 'mean'),
    coverage_mean=('cellwise_95_coverage', 'mean'),
    n_worlds=('replicate', 'nunique'),
)
final_summary.to_csv(OUTPUT_DIR / 'service_extension_final_summary.csv', index=False)

primary_vs_unweighted = paired_comparisons.loc[
    (paired_comparisons.target_name == 'primary_equal_service')
    & (paired_comparisons.baseline == 'unweighted_aopt')
].iloc[0]
primary_vs_fixed = paired_comparisons.loc[
    (paired_comparisons.target_name == 'primary_equal_service')
    & (paired_comparisons.baseline == 'best_fixed_target')
].iloc[0]
primary_policy_summary = final_summary.loc[
    (final_summary.target_name == 'primary_equal_service')
    & (final_summary.policy == 'target_aopt')
].iloc[0]

primary_actions = action_history.loc[action_history.target_name == 'primary_equal_service']
target_sequences = primary_actions.loc[primary_actions.policy == 'target_aopt'].sort_values(['replicate', 'step'])
unweighted_sequences = primary_actions.loc[primary_actions.policy == 'unweighted_aopt'].sort_values(['replicate', 'step'])
sequence_pairs = target_sequences.merge(
    unweighted_sequences, on=['replicate', 'step'], suffixes=('_target', '_unweighted'), validate='one_to_one'
)
sequence_difference = (
    sequence_pairs.action_index_target.ne(sequence_pairs.action_index_unweighted)
    .groupby(sequence_pairs.replicate).any()
)

gate_rows = [
    {
        'gate': 'service_aware_beats_unweighted_adaptive_primary',
        'passed': bool(primary_vs_unweighted.uncertainty_improvement_ci95_low > 0),
        'observed': float(primary_vs_unweighted.uncertainty_improvement_ci95_low),
        'criterion': 'primary paired-bootstrap lower 95% CI > 0%',
    },
    {
        'gate': 'service_aware_beats_target_matched_best_fixed',
        'passed': bool(primary_vs_fixed.uncertainty_improvement_ci95_low > 0),
        'observed': float(primary_vs_fixed.uncertainty_improvement_ci95_low),
        'criterion': 'secondary paired-bootstrap lower 95% CI > 0%',
    },
    {
        'gate': 'primary_rmse_noninferior_to_unweighted',
        'passed': bool(
            primary_vs_unweighted.target_rmse_difference_ci95_high
            < 0.05 * final_summary.loc[
                (final_summary.target_name == 'primary_equal_service')
                & (final_summary.policy == 'unweighted_aopt'), 'rmse_mean'
            ].iloc[0]
        ),
        'observed': float(primary_vs_unweighted.target_rmse_difference_ci95_high),
        'criterion': 'upper 95% CI for RMSE harm < 5% of unweighted RMSE',
    },
    {
        'gate': 'primary_cellwise_coverage_reasonable',
        'passed': bool(0.90 <= primary_policy_summary.coverage_mean <= 0.98),
        'observed': float(primary_policy_summary.coverage_mean),
        'criterion': '0.90 <= mean empirical 95% coverage <= 0.98',
    },
    {
        'gate': 'at_least_three_cswim_services_represented',
        'passed': bool(len(present_services) >= 3),
        'observed': int(len(present_services)),
        'criterion': 'at least three services have nonzero exposure in the modeled window',
    },
    {
        'gate': 'current_catalog_classification_match_rate_at_least_half',
        'passed': bool(satellite_match_rate >= 0.50),
        'observed': float(satellite_match_rate),
        'criterion': 'at least 50% of in-window active satellites matched to the frozen C-SWIM catalog',
    },
]
claim_gates = pd.DataFrame(gate_rows)
claim_gates.to_csv(OUTPUT_DIR / 'service_extension_claim_gates.csv', index=False)

claim_status = {
    'service_layer_superiority_supported': bool(claim_gates.loc[
        claim_gates.gate == 'service_aware_beats_unweighted_adaptive_primary', 'passed'
    ].iloc[0]),
    'service_policy_beats_target_matched_fixed': bool(claim_gates.loc[
        claim_gates.gate == 'service_aware_beats_target_matched_best_fixed', 'passed'
    ].iloc[0]),
    'all_primary_service_claim_gates_pass': bool(claim_gates.passed.all()),
    'fraction_worlds_schedule_differs_from_unweighted': float(sequence_difference.mean()),
    'classification_match_rate': satellite_match_rate,
    'present_services': present_services,
    'absent_services': [s for s in SERVICE_CATEGORIES if s not in present_services],
    'interpretation_if_primary_fails': (
        'Keep the C-SWIM layer as an exposure/sensitivity analysis, but do not claim it improves '
        'adaptive scheduling beyond the unweighted policy.'
    ),
}
(OUTPUT_DIR / 'service_extension_claim_status.json').write_text(json.dumps(claim_status, indent=2))

print('PRIMARY SERVICE-EXTENSION RESULT')
print(
    f"  versus unweighted adaptive: {primary_vs_unweighted.uncertainty_improvement_percent:.2f}% "
    f"(95% CI {primary_vs_unweighted.uncertainty_improvement_ci95_low:.2f}% to "
    f"{primary_vs_unweighted.uncertainty_improvement_ci95_high:.2f}%)"
)
print(
    f"  versus target-matched fixed: {primary_vs_fixed.uncertainty_improvement_percent:.2f}% "
    f"(95% CI {primary_vs_fixed.uncertainty_improvement_ci95_low:.2f}% to "
    f"{primary_vs_fixed.uncertainty_improvement_ci95_high:.2f}%)"
)
print('  Service superiority supported:', claim_status['service_layer_superiority_supported'])
display(claim_gates)


## 7. Candidate manuscript panels


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(13.2, 9.2), constrained_layout=True)

ax = axes[0, 0]
for i, service in enumerate(SERVICE_CATEGORIES):
    if service_profiles[i].sum() > 0:
        ax.plot(ALTITUDE_CENTERS, 100 * service_profiles[i], marker='o', linewidth=2, label=service)
ax.set(title='A. C-SWIM service exposure by orbital shell', xlabel='Altitude shell center (km)', ylabel='Within-service exposure (%)')
ax.legend(frameon=False, ncol=2)

ax = axes[0, 1]
ax.plot(ALTITUDE_CENTERS, 100 * PRIMARY_SERVICE_WEIGHTS, marker='o', linewidth=2.3, label='Equal-service target')
ax.plot(ALTITUDE_CENTERS, 100 * ALL_ACTIVE_WEIGHTS, marker='s', linewidth=1.8, label='All active satellites')
ax.axhline(10, color='black', linestyle=':', linewidth=1, label='Equal shell weight')
ax.set(title='B. Predeclared service target changes the emphasis', xlabel='Altitude shell center (km)', ylabel='Target weight (%)')
ax.legend(frameon=False)

ax = axes[1, 0]
primary_steps = step_metrics.loc[step_metrics.target_name == 'primary_equal_service']
for policy in POLICIES:
    group = primary_steps.loc[primary_steps.policy == policy].groupby('step').target_uncertainty_fraction
    mean = group.mean(); se = group.std(ddof=1) / np.sqrt(group.count())
    ax.plot(mean.index, mean, color=POLICY_COLORS[policy], linewidth=2, label=POLICY_LABELS[policy])
    ax.fill_between(mean.index, mean - 1.96 * se, mean + 1.96 * se, color=POLICY_COLORS[policy], alpha=0.10)
ax.set(title='C. Equal-service uncertainty remaining', xlabel='One-hour observations', ylabel='Weighted uncertainty fraction')
ax.legend(frameon=False, fontsize=8)

ax = axes[1, 1]
forest = paired_comparisons.loc[paired_comparisons.baseline == 'unweighted_aopt'].copy()
forest['primary_order'] = (forest.target_name == 'primary_equal_service').astype(int)
forest = forest.sort_values(['primary_order', 'target_name'])
y = np.arange(len(forest))
for j, row in enumerate(forest.itertuples(index=False)):
    color = '#E45756' if row.target_name == 'primary_equal_service' else '#4C78A8'
    ax.errorbar(
        row.uncertainty_improvement_percent, j,
        xerr=[[row.uncertainty_improvement_percent - row.uncertainty_improvement_ci95_low],
              [row.uncertainty_improvement_ci95_high - row.uncertainty_improvement_percent]],
        fmt='o', color=color, ecolor=color, capsize=3,
    )
ax.axvline(0, color='black', linewidth=1)
ax.set_yticks(y, forest.display_label)
ax.set(title='D. Service-aware versus unweighted adaptive', xlabel='Weighted uncertainty reduction after 24 h (%)')
ax.grid(axis='y', visible=False)

fig.suptitle('C-SWIM service-aware adaptive radar scheduling', fontsize=15, fontweight='bold')
fig.savefig(OUTPUT_DIR / 'figure_cswim_service_extension_main_2x2.png', dpi=220, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_cswim_service_extension_main_2x2.pdf', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12.8, 8.8), constrained_layout=True)
ax = axes[0, 0]
counts = in_window.service.value_counts().reindex(list(SERVICE_CATEGORIES) + ['OTHER', 'UNCLASSIFIED'], fill_value=0)
bars = ax.bar(counts.index, counts.values, color=['#4C78A8'] * 5 + ['#999999', '#E45756'])
ax.bar_label(bars, fontsize=8)
ax.tick_params(axis='x', rotation=35)
ax.set(title='A. In-window active satellites by audited class', ylabel='Unique satellites')

ax = axes[0, 1]
ax.bar(['C-SWIM matched', 'Unclassified'], [satellite_match_rate, 1 - satellite_match_rate], color=['#4C78A8', '#E45756'])
ax.set_ylim(0, 1)
ax.set(title='B. Current-catalog classification coverage', ylabel='Share of in-window satellites')

ax = axes[1, 0]
action_counts = primary_actions.groupby(['policy', 'elevation_deg']).size().unstack(fill_value=0)
for policy in ('target_aopt', 'unweighted_aopt'):
    values = action_counts.loc[policy] / action_counts.loc[policy].sum()
    ax.plot(values.index, values.values, marker='o', linewidth=2, label=POLICY_LABELS[policy])
ax.set(title='C. Adaptive action use under the primary target', xlabel='Elevation (deg)', ylabel='Share of observations')
ax.legend(frameon=False)

ax = axes[1, 1]
primary_final = final_metrics.loc[final_metrics.target_name == 'primary_equal_service']
service_columns = [f'{s.lower()}_uncertainty_fraction' for s in present_services]
service_policy_means = primary_final.groupby('policy')[service_columns].mean()
x = np.arange(len(present_services)); width = 0.36
ax.bar(x - width/2, service_policy_means.loc['unweighted_aopt'], width, label='Unweighted adaptive', color='#F2A541')
ax.bar(x + width/2, service_policy_means.loc['target_aopt'], width, label='Service-aware adaptive', color='#E45756')
ax.set_xticks(x, present_services)
ax.set(title='D. Final uncertainty within each service', ylabel='Uncertainty fraction')
ax.legend(frameon=False)

fig.suptitle('C-SWIM classification and policy diagnostics', fontsize=15, fontweight='bold')
fig.savefig(OUTPUT_DIR / 'figure_cswim_service_extension_diagnostics_2x2.png', dpi=220, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_cswim_service_extension_diagnostics_2x2.pdf', bbox_inches='tight')
plt.show()


## 8. Reproducibility manifest and plain-language status


In [ ]:
readme = f"# C-SWIM service-extension run `{config_hash}`\n\nThis run tests a predeclared service-aware extension of the completed adaptive Bayesian scheduler.\n\n- Service labels come from C-SWIM v1 (DOI 10.5281/zenodo.19477756).\n- The primary target gives each represented C-SWIM service equal total influence and uses equal satellite weights within service.\n- No subjective ranking of navigation, communications, weather, Earth observation, or military/intelligence is used.\n- C-SWIM replacement value is a labeled sensitivity only.\n- Unmatched current satellites remain unclassified and are included as a separate-group sensitivity.\n- The confirmatory response uses ORDEM >1 cm, consistent with the published EISCAT description of\n  sensitivity down to a few centimeters; ORDEM >1 mm is a named sensitivity only.\n- The only confirmatory extension contrast is service-aware A-opt versus unweighted A-opt on the equal-service endpoint.\n\nPrimary improvement versus unweighted adaptive:\n{primary_vs_unweighted.uncertainty_improvement_percent:.2f}%\n(95% paired-bootstrap CI {primary_vs_unweighted.uncertainty_improvement_ci95_low:.2f}% to\n{primary_vs_unweighted.uncertainty_improvement_ci95_high:.2f}%).\n\nService-layer superiority supported: {claim_status['service_layer_superiority_supported']}.\n\n`_SUCCESS.json` means the notebook completed. It does not mean the service-superiority claim passed.\nIndependent Svalbard validation remains a separate empirical requirement.\n"
(OUTPUT_DIR / 'README_cswim_service_extension.md').write_text(readme)

def output_manifest_rows():
    rows = []
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file() and path.name not in {'output_manifest.json', '_SUCCESS.json'}:
            rows.append({'file': path.name, 'bytes': path.stat().st_size, 'sha256': sha256_file(path)})
    return rows
manifest = {'config_hash': config_hash, 'created_utc': datetime.now(timezone.utc).isoformat(), 'input_hashes': input_hashes, 'outputs': output_manifest_rows()}
(OUTPUT_DIR / 'output_manifest.json').write_text(json.dumps(manifest, indent=2))
success = {'success': True, 'config_hash': config_hash, 'output_directory': str(OUTPUT_DIR), 'service_layer_superiority_supported': claim_status['service_layer_superiority_supported'], 'service_policy_beats_target_matched_fixed': claim_status['service_policy_beats_target_matched_fixed'], 'classification_match_rate': satellite_match_rate, 'present_services': present_services, 'files_in_manifest': len(manifest['outputs'])}
(OUTPUT_DIR / '_SUCCESS.json').write_text(json.dumps(success, indent=2))
print('\nSUCCESS — C-SWIM service-extension run is complete.')
print(json.dumps(success, indent=2))
